# 23b · Enriched factor model: order book, taker flow, derivatives, ETH and sentiment changes (E23 step 2)

The agent's 8 factors come from E15's GRU, which reads only 1-second spot klines (alpha101, intra-minute statistics
and their slow EMAs). An IC scan on train + validation found small but stable signals in data the GRU never saw
(`rltrade.enrich`): futures order-book imbalance and depth, taker flow over 1 h - 1 day, Binance funding rate, open
interest and long/short ratios, ETH returns, and 1-day changes of the LLM sentiment / risk scores. A linear model on
all of them reached a validation IC of about 0.10 (5 h) and 0.18 (1 day), against 0.13 for E15's f5.

This notebook adds those **32 inputs** to E15's input set and retrains the GRU with **E15's exact settings**
(same hyperparameters, 3 seeds averaged, early stopping on validation), so the only change is the inputs. Then the
train period is **cross-fitted** as in E20a (4 blocks, 1-day embargo), so the RL agent can later train on honest
train factors.

**Input:** upload `e23_extra_inputs.zip` (built locally by `python -m rltrade.enrich raw ...`, about 150 MB) to
`thesis_rl/initiation/`. It holds unscaled feature arrays for every split, row-aligned with the prepared set; they
are scaled here with train statistics only. Test and stress_bear are predicted (the GRU reads one continuous series)
but **no metric on them is computed or shown**.

T4 GPU, about 45-60 min (15 GRU runs of 2-4 min). Runs resume after a disconnect. Writes
`thesis_rl/initiation/e23_factors/` (`selected_mean3/`, `crossfit_mean3/`) and `report/e23b_*`.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/14-initiation"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "23b_enriched_factors")

In [ ]:
import json, zipfile, numpy as np, pandas as pd
from pathlib import Path
from rltrade import long_context, enrich, factor_rule as FR
from rltrade.rnn import tune_gru as TG, crossfit as CF
from rltrade.rnn.train import label_metrics, available_splits

INIT = L.root / "initiation"
E15, OUT, REP = INIT / "e15", INIT / "e23_factors", INIT / "report"
for p in (OUT, REP): p.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/initiation_prepared") if "google.colab" in sys.modules else INIT / "prepared"
ZIP, XZIP = INIT / "initiation_prepared.zip", INIT / "e23_extra_inputs.zip"
if not (LOCAL / "fixed_plus" / "meta.json").exists():
    assert ZIP.exists(), f"upload initiation_prepared.zip to {ZIP}"
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL)
if not (LOCAL / "fixed_plus_long" / "meta.json").exists():
    long_context.build(LOCAL / "fixed_plus", LOCAL / "fixed_plus_long")
if not (LOCAL / "e23_extra" / "extra.json").exists():
    assert XZIP.exists(), f"upload e23_extra_inputs.zip to {XZIP}"
    with zipfile.ZipFile(XZIP) as z: z.extractall(LOCAL)

def on_drive(p):                                    # paths written on another machine: same place under initiation/
    p = Path(p)
    return p if p.exists() else INIT.joinpath(*p.parts[p.parts.index("initiation") + 1:])
PARAMS = json.loads((E15 / "best_params.json").read_text())
INPUTS = PARAMS["inputs"]                           # E15's input set; the enriched set = it + the extra inputs
ENR = LOCAL / f"{INPUTS}_e23"
if not (ENR / "meta.json").exists():
    enrich.build(LOCAL / INPUTS, LOCAL / "e23_extra", ENR)
EXTRA = json.loads((ENR / "meta.json").read_text())["enrich"]["columns"]
SPLITS = available_splits(ENR)
N_TRAIN = len(np.load(ENR / "alpha101_train.npy", mmap_mode="r"))
E15_SEL = on_drive(json.loads((INIT / "selected_factors_e15.json").read_text())["factor_dir"])
E20_CF = INIT / "e20" / "crossfit_mean3"
SEEDS, K = (0, 1, 2), 4
PRICE = {s: np.load(LOCAL / "fixed_plus" / f"market_{s}.npz")["midpoint"].astype(np.float64) for s in ("train", "val")}
Y = {s: np.load(ENR / f"labels_{s}.npy") for s in ("train", "val")}
print(f"E15 config: {PARAMS}\ninputs: {INPUTS} + {len(EXTRA)} extra -> {ENR.name} ({json.loads((ENR / 'meta.json').read_text())['n_inputs']} inputs)")

## 1 · Each extra input on its own: IC vs the forward return (train and validation)

In [ ]:
def fwd_ic(x, split, h):
    return FR.ic(np.asarray(x, np.float64), FR.forward_returns(PRICE[split], h), h)[0]
X = {s: np.load(ENR / f"alpha101_{s}.npy", mmap_mode="r") for s in ("train", "val")}
n0 = X["train"].shape[1] - len(EXTRA)
rows = []
for j, c in enumerate(EXTRA):
    r = {"input": c}
    for s in ("train", "val"):
        for h in (60, 300, 900):
            r[f"{s} {h}"] = fwd_ic(X[s][:, n0 + j], s, h)
    rows.append(r)
single = pd.DataFrame(rows)
single.to_csv(OUT / "extra_input_ic.csv", index=False)
print("Spearman IC of each extra input vs the forward return over 60 / 300 / 900 bars (non-overlapping samples):")
print(single.round(3).to_string(index=False))

## 2 · Retrain E15's GRU on the enriched inputs (3 seeds, E15 settings)

In [ ]:
_, FINAL = TG.final_runs(PARAMS, {INPUTS: ENR}, OUT, seeds=SEEDS)
SEL = TG.write_mixture(FINAL, [], 1.0, OUT / "selected_mean3", SPLITS)
runs = pd.DataFrame([{"seed": s, **{k: json.loads((d / "metrics.json").read_text())[k] for k in ("steps_run", "stop_reason", "minutes")},
                      "val mse": json.loads((d / "metrics.json").read_text())["val"]["mse"]} for s, d in zip(SEEDS, FINAL)])
print(runs.round(5).to_string(index=False))

def factor_table(fdir, split, tag):
    f = np.load(Path(fdir) / f"factors_{split}.npy")
    m = label_metrics(f, Y[split]) if split == "val" else None
    r = {"factors": tag, "split": split}
    if m is not None:
        r.update({"label mse": m["mse"], "label ic": m["ic_mean"]})
    for j, h in enumerate(FR.OFFSETS):
        r[f"f{j} ({h})"] = fwd_ic(f[:, j], split, h)
    return r
cmp = pd.DataFrame([factor_table(E15_SEL, "val", "E15 GRU (current)"), factor_table(SEL, "val", "E23 enriched GRU")])
print("\nvalidation: label fit, and IC of each factor vs the forward return over its own horizon (bars):")
print(cmp.round(4).to_string(index=False))

## 3 · Cross-fit the train period (as E20a: 4 blocks, 1-day embargo, 3 seeds)

In [ ]:
FOLDS = CF.crossfit_runs(PARAMS, ENR, OUT / "crossfit", N_TRAIN, k=K, seeds=SEEDS)
CFD = CF.write_factors(FOLDS, N_TRAIN, SEL, OUT / "crossfit_mean3", SPLITS, k=K)
rows = []
for tag, fd in (("E20a cross-fitted (current)", E20_CF), ("E23 enriched, cross-fitted", CFD)):
    if not (Path(fd) / "factors_train.npy").exists(): continue
    f = np.load(Path(fd) / "factors_train.npy")
    for b, (lo, hi) in enumerate(CF.blocks(N_TRAIN, K)):
        r = {"factors": tag, "train block": b}
        for j in (3, 5, 7):
            h = FR.OFFSETS[j]
            r[f"f{j} ({h}) IC"] = FR.ic(f[lo:hi, j].astype(np.float64), FR.forward_returns(PRICE["train"][lo:hi], h), h)[0]
        rows.append(r)
cf_cmp = pd.DataFrame(rows)
print("train, out-of-sample (each block predicted by models that never saw it):")
print(cf_cmp.round(3).to_string(index=False))
print("\nmean over blocks:\n" + cf_cmp.groupby("factors").mean(numeric_only=True).drop(columns="train block").round(3).to_string())

In [ ]:
(INIT / "selected_factors_e23.json").write_text(json.dumps({"factor_dir": str(SEL), "crossfit_dir": str(CFD),
    "inputs": ENR.name, "extra_inputs": EXTRA, "params": PARAMS, "seeds": list(SEEDS),
    "note": "E15 settings on E15 inputs + E23 extra inputs; train factors cross-fitted in crossfit_dir"}, indent=1))
(REP / "e23b_enriched_factors.md").write_text(
    "# E23b: enriched factor model (train + validation only)\n\n"
    f"E15 GRU settings ({PARAMS}) on {INPUTS} + {len(EXTRA)} extra inputs. Mean of seeds {SEEDS}.\n\n"
    "## Validation: label fit and factor IC vs forward return\n\n" + cmp.round(4).to_markdown(index=False)
    + "\n\n## Train, cross-fitted, by block\n\n" + cf_cmp.round(3).to_markdown(index=False)
    + "\n\n## Each extra input alone\n\n" + single.round(3).to_markdown(index=False)
    + "\n\n## GRU runs\n\n" + runs.round(5).to_markdown(index=False) + "\n", encoding="utf-8")
print("wrote", REP / "e23b_enriched_factors.md", "|", INIT / "selected_factors_e23.json")

In [ ]:
AUTO_DISCONNECT = True               # release the GPU once the results are on Drive
if AUTO_DISCONNECT and "google.colab" in sys.modules:
    from google.colab import runtime
    runtime.unassign()